In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path(r"C:\Data Engineering\Projects\fleetpulse_project")

if str(PROJECT_ROOT) is not sys.path:
    sys.path.append(str(PROJECT_ROOT))

In [2]:
from scripts.utils.spark_session import create_delta_spark_session
from scripts.utils.config import (
    BRONZE_BASE_PATH,
    SILVER_BASE_PATH,
    GOLD_BASE_PATH
)

In [3]:
spark = create_delta_spark_session(app_name="DeltaTableTest")



In [4]:
data = [
    (101, "BR-001", "Downtown Branch", "North", "Active"),
    (102, "BR-002", "Airport Hub", "East", "Active"),
    (103, "BR-003", "Westside Depot", "West", "Maintenance")
]
columns = ["branch_key", "branch_id", "branch_name", "region", "branch_status"]

df = spark.createDataFrame(data, columns)
df.show(truncate=False)

+----------+---------+---------------+------+-------------+
|branch_key|branch_id|branch_name    |region|branch_status|
+----------+---------+---------------+------+-------------+
|101       |BR-001   |Downtown Branch|North |Active       |
|102       |BR-002   |Airport Hub    |East  |Active       |
|103       |BR-003   |Westside Depot |West  |Maintenance  |
+----------+---------+---------------+------+-------------+



In [5]:
table_name = "__test_delta"
gold_path = f"{GOLD_BASE_PATH}/{table_name}"
df.write.mode("overwrite").format("delta").save(gold_path)

In [6]:
read_df = spark.read.format('delta').load(gold_path)
read_df.show(truncate=False)

+----------+---------+---------------+------+-------------+
|branch_key|branch_id|branch_name    |region|branch_status|
+----------+---------+---------------+------+-------------+
|101       |BR-001   |Downtown Branch|North |Active       |
|102       |BR-002   |Airport Hub    |East  |Active       |
|103       |BR-003   |Westside Depot |West  |Maintenance  |
+----------+---------+---------------+------+-------------+



In [7]:
from scripts.config.gold_table_config import GOLD_TABLE_CONFIG

dimension_config = GOLD_TABLE_CONFIG
print(dimension_config['dim_branch']['business_columns'])

['branch_id', 'branch_code', 'branch_name', 'region', 'district', 'city', 'state', 'country', 'postal_code', 'latitude', 'longitude', 'branch_status', 'source_system', 'record_hash']


In [8]:
from scripts.utils.gold_utils import build_incoming_dimension

df = build_incoming_dimension(spark=spark,dimension_config=GOLD_TABLE_CONFIG['dim_branch'],silver_base_path=SILVER_BASE_PATH)
df.show()

+---------+-----------+--------------------+---------+-------------+----------+-----+-------+-----------+---------+----------+-------------+-------------+--------------------+
|branch_id|branch_code|         branch_name|   region|     district|      city|state|country|postal_code| latitude| longitude|branch_status|source_system|         record_hash|
+---------+-----------+--------------------+---------+-------------+----------+-----+-------+-----------+---------+----------+-------------+-------------+--------------------+
|        1|  BR-OH-001|Youngstown Rental...|  Midwest|  Ohio Valley|Youngstown|   OH|    USA|      44503|41.099800|-80.649500|       ACTIVE|  RENTAL_OLTP|87b64ae6e31523aba...|
|        2|  BR-OH-002|Cleveland Equipme...|  Midwest|Northern Ohio| Cleveland|   OH|    USA|      44114|41.499300|-81.694400|       ACTIVE|  RENTAL_OLTP|5fbd7c136f3bea57e...|
|        3|  BR-PA-001|Pittsburgh Indust...|Northeast|   Western PA|Pittsburgh|   PA|    USA|      15222|40.440600|-79.9

In [11]:

from scripts.utils.gold_utils import incremental_load
gold_table_name = dimension_config["dim_branch"]["gold_table_name"]
silver_table_name = dimension_config["dim_branch"]["source_table"]
silver_path = f"{SILVER_BASE_PATH}/{silver_table_name}"
gold_path = f"{GOLD_BASE_PATH}/{gold_table_name}"
silver_df = spark.read.parquet(silver_path)
new_record_data = [{
    "branch_id" : "1",
    "branch_name" : "Youngstown Rental Center Updated",
    "record_hash" : "f4c34e34a67130e53"
},
{   "branch_id" : "256",
    "branch_name" : "Youngstown Rental Center Updated with 2",
    "record_hash" : "f4c34e34a67130e53"}]
new_record_df = spark.createDataFrame(new_record_data)
updated_silver_df = silver_df.unionByName(new_record_df, allowMissingColumns=True)
gold_df = spark.read.format('delta').load(gold_path)
results = incremental_load(incoming_df=updated_silver_df,gold_df=gold_df,table_config=dimension_config["dim_branch"])
print(results["counts"])
results["changed_df"].show(truncate = False)
results["new_df"].show(truncate = False)
results["rows_to_insert_df"].show(truncate = False)
results["rows_to_insert_with_keys_df"].show(truncate = False)

{'incoming_count': 6, 'current_dim_count': 4, 'unchanged_count': 4, 'changed_count': 1, 'new_count': 1, 'rows_to_insert_count': 2}
+---------+-----------+--------------------------------+------+--------+----+-----+-------+-----------+--------+---------+-------------+-------------+----------+----------+----------+-------------------+--------------+------------+----------------+-----------------+------------------------+------------------------+---------------------+
|branch_id|branch_code|branch_name                     |region|district|city|state|country|postal_code|latitude|longitude|branch_status|source_system|created_at|updated_at|is_deleted|ingestion_timestamp|ingestion_date|source_table|bronze_load_type|record_hash      |silver_created_timestamp|silver_updated_timestamp|silver_quality_status|
+---------+-----------+--------------------------------+------+--------+----+-----+-------+-----------+--------+---------+-------------+-------------+----------+----------+----------+--------